In [3]:
import boto3
import os
from botocore import UNSIGNED
from botocore.config import Config
import xarray
import eccodes


In [4]:
import isodate
from datetime import datetime

model = 'HARMONIE_DINI_SF'
model_run = '2026-06-15T000000Z'
modeldir = 'modeldir'

forecast_timestep = "PT1H"
forecast_length = "PT2H"

# Create the model directory if it does not exist
os.makedirs(modeldir, exist_ok=True)

# Initialize the S3 client
config = Config(signature_version=UNSIGNED)
s3 = boto3.client('s3', region_name='eu-north-1', config=config)

# Bucket where modelfiles are located
bucket_name = 'dmi-opendata'

model_run_dt = datetime.strptime(model_run, '%Y-%m-%dT%H%M%SZ')
timestep_td = isodate.parse_duration(forecast_timestep)
length_td = isodate.parse_duration(forecast_length)

valid_times = []
current = model_run_dt
while current <= model_run_dt + length_td:
    valid_times.append(current)
    current += timestep_td

for vt in valid_times:
    vt_str = vt.strftime('%Y-%m-%dT%H%M%SZ')
    key = f"forecastdata/{model}/{model}_{model_run}_{vt_str}.grib"
    localfile = key.split('/')[-1]
    if not os.path.exists(os.path.join(modeldir, localfile)):
        print(f'Downloading {key}...')
        s3.download_file(bucket_name, key, os.path.join(modeldir, localfile))
        print(f'{key} downloaded successfully.')
    else:
        print(f'{localfile} already exists. Skipping download.')


HARMONIE_DINI_SF_2026-06-15T000000Z_2026-06-15T000000Z.grib already exists. Skipping download.
HARMONIE_DINI_SF_2026-06-15T000000Z_2026-06-15T010000Z.grib already exists. Skipping download.
HARMONIE_DINI_SF_2026-06-15T000000Z_2026-06-15T020000Z.grib already exists. Skipping download.


In [ ]:
import eccodes
import numpy as np
import xarray as xr

filter_keys = {
    'discipline': 0,
    'parameterCategory': 0,
    'parameterNumber': 0,
    'level': 2,
    'typeOfFirstFixedSurface': 103,
}

arrays = []
lats_2d = None
lons_2d = None

for vt in valid_times:
    vt_str = vt.strftime('%Y-%m-%dT%H%M%SZ')
    gribfile = os.path.join(modeldir, f"{model}_{model_run}_{vt_str}.grib")

    with open(gribfile, 'rb') as f:
        while True:
            msg = eccodes.codes_grib_new_from_file(f)
            if msg is None:
                break
            try:
                if all(eccodes.codes_get(msg, k) == v for k, v in filter_keys.items()):
                    print(f"Found matching message for valid time {vt_str}")
                    try:
                        nx = eccodes.codes_get(msg, 'Ni')
                        ny = eccodes.codes_get(msg, 'Nj')
                    except eccodes.KeyValueNotFoundError:
                        nx = eccodes.codes_get(msg, 'Nx')
                        ny = eccodes.codes_get(msg, 'Ny')
                    values = eccodes.codes_get_values(msg).reshape(ny, nx)
                    if lats_2d is None:
                        lats_2d = eccodes.codes_get_array(msg, 'latitudes').reshape(ny, nx)
                        lons_2d = eccodes.codes_get_array(msg, 'longitudes').reshape(ny, nx)
                    arrays.append(values)
                    break
            finally:
                eccodes.codes_release(msg)

t2m = xr.Dataset(
    {'t2m': (['valid_time', 'y', 'x'], np.stack(arrays))},
    coords={
        'valid_time': valid_times,
        'latitude': (['y', 'x'], lats_2d),
        'longitude': (['y', 'x'], lons_2d),
    }
)
t2m


ValueError: need at least one array to stack